# Understading features

Information Geometry Soft Clustering for Sport Analytics

Joaquin Garay

In [1]:
import pandas as pd
import socceraction.spadl as spadl
import matplotlib.pyplot as plt
import os

In [2]:
datafolder = "../data"
fifa2018h5 = os.path.join(datafolder, "spadl-fifa2018.h5")

### What's inside?

In [3]:
with pd.HDFStore(fifa2018h5) as store:
    print(store.keys())

FileNotFoundError: ``/Users/jgv/PycharmProjects/cylindrical-data-lvm/experiment/data`` does not exist

In [ ]:
#Competitions
pd.read_hdf(fifa2018h5, key="competitions").head()

In [ ]:
#games
pd.read_hdf(fifa2018h5, key="games")

In [ ]:
#player games
pd.read_hdf(fifa2018h5, key="player_games").head()

In [ ]:
#players
pd.read_hdf(fifa2018h5, key="players").head()

In [ ]:
#teams
pd.read_hdf(fifa2018h5, key="teams").head()
#France ID = 771

In [ ]:
#actions
pd.read_hdf(fifa2018h5, key="actions/game_7525").head()

### Location of actions

Like mentioned in SoccerMix paper, the locations in event stream data are approximations. For some actions, such as goal kicks and crosses, annotators use a set of predefined start locations instead of its actual location. It is possible to note those patterns on the following plots.

In [ ]:
datafolder = "../data"
fifa2018h5 = os.path.join(datafolder, "spadl-fifa2018.h5")
games = pd.read_hdf(fifa2018h5, key="games")
with pd.HDFStore(fifa2018h5) as store:
    actions = [] #list of DataFrames
    for game in games.itertuples():
        game_action = store[f"actions/game_{game.game_id}"]
        game_action = spadl.play_left_to_right(game_action, game.home_team_id)
        game_action["is_home"] = game_action["team_id"] == game.home_team_id
        actions.append(game_action)
    actions = pd.concat(actions)
    actions.drop("original_event_id", axis=1, inplace=True )
    actions = pd.merge(actions, spadl.config.actiontypes_df(), how="left")

In [ ]:
set(actions["type_name"])

In [ ]:
for actiontype in set(actions["type_name"]):
    actions[actions.type_name == actiontype].plot.scatter(
        x="start_x",
        y="start_y",
        title = f"Start Location: {actiontype}",
        figsize = (6,4)
    )
    plt.show()
    actions[actions.type_name == actiontype].plot.scatter(
        x="end_x",
        y="end_y",
        title = f"End Location: {actiontype}",
        figsize = (6,4)
    )
    plt.show()

In [ ]:
def consolidate(actions):
    #actions.fillna(0, inplace=True)

    #Consolidate corner_short and corner_crossed
    corner_idx = actions.type_name.str.contains("corner")
    actions["type_name"] = actions["type_name"].mask(corner_idx,"corner")

    #Consolidate freekick_short, freekick_crossed, and shot_freekick
    freekick_idx = actions.type_name.str.contains("freekick")
    actions["type_name"] = actions["type_name"].mask(freekick_idx,"freekick")

    #Consolidate keeper_claim, keeper_punch, keeper_save, keeper_pick_up
    keeper_idx = actions.type_name.str.contains("keeper")
    actions["type_name"] = actions["type_name"].mask(keeper_idx,"keeper_action")

    actions["start_x"] = actions["start_x"].mask(actions.type_name=="shot_penalty",94.5)
    actions["start_y"] = actions["start_y"].mask(actions.type_name=="shot_penalty",34)

    return actions

actions = consolidate(actions)
for actiontype in ["corner", "freekick", "keeper_action"]:
    actions[actions.type_name == actiontype].plot.scatter(
        x="start_x",
        y="start_y",
        title = f"{actiontype}"
    )
    plt.show()

### Analyzing one match

In [ ]:
match = pd.read_hdf(fifa2018h5, key="actions/game_7585").drop("original_event_id", axis=1)
actiontypes_df = spadl.config.actiontypes_df()
results_df = spadl.results_df()
bodyparts_df = spadl.bodyparts_df()

In [ ]:
match.describe()

In [ ]:
actiontypes_df.head()

In [ ]:
#Dribbles
match[match["type_id"] == 21 ]

In [ ]:
#two full-time halfs, and two 15-min suplementary halfs. timeperiod==5 should be penalties.
df_aux = match.groupby("period_id")
df_aux["time_seconds"].plot(kind="hist")